*0.1 Python for GenAI*

# httpx

**The situation.** Your company adds an internal model server. It speaks a slightly different protocol and has no SDK. Separately, during an incident, nobody can say what the OpenAI SDK actually sends — which headers, what timeout, how many retries — because it has always been a black box.

**What the SDK really does.** It sends one web request: a POST to a URL, with your key in a header and a JSON body. Back comes JSON and a *status code* (200 ok, 404 not found, 429 slow down, 500 server error). `httpx` is the library that sends that request — the SDK is a thin layer on top of it.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**A production client.** Built once, reused for every call: pooled connections, a bearer token, and every timeout set explicitly — connect, read, write, pool.

In [2]:
import json
import os

import httpx

http = httpx.Client(
    base_url="https://api.openai.com/v1",
    headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
    timeout=httpx.Timeout(connect=5.0, read=30.0, write=10.0, pool=5.0),
    transport=httpx.HTTPTransport(retries=2),  # retry connection failures
)
body = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Reply with the single word: pong"}],
    "max_tokens": 3,
}

response = http.post("/chat/completions", json=body)
response.raise_for_status()  # turn 4xx / 5xx into an exception
print(
    "status:", response.status_code, "| elapsed:", round(response.elapsed.total_seconds(), 2), "s"
)
print("reply:", response.json()["choices"][0]["message"]["content"])
print("usage:", json.dumps(response.json()["usage"], indent=2))
print("rate limit left this minute:", response.headers.get("x-ratelimit-remaining-requests"))
assert "pong" in response.json()["choices"][0]["message"]["content"].lower()

status: 200 | elapsed: 0.89 s
reply: Pong
usage: {
  "prompt_tokens": 14,
  "completion_tokens": 2,
  "total_tokens": 16,
  "prompt_tokens_details": {
    "cached_tokens": 0,
    "audio_tokens": 0
  },
  "completion_tokens_details": {
    "reasoning_tokens": 0,
    "audio_tokens": 0,
    "accepted_prediction_tokens": 0,
    "rejected_prediction_tokens": 0
  }
}
rate limit left this minute: 9999


**Reading the output.** Status 200, the reply, the token usage the SDK normally hides in `.usage`, and a header the SDK reads for you: how many requests you have left this minute.

**The failure path.** Ask for a model that does not exist.

In [3]:
bad = http.post("/chat/completions", json={**body, "model": "no-such-model"})
try:
    bad.raise_for_status()
except httpx.HTTPStatusError as error:
    print(
        "status:", error.response.status_code, "→", error.response.json()["error"]["message"][:60]
    )
    bad_status = error.response.status_code
http.close()
assert bad_status == 404

status: 404 → The model `no-such-model` does not exist or you do not have 


**Reading the output.** A 404 with the provider's own explanation in the body. This is what the SDK turns into `NotFoundError`.

```
your code ──POST /v1/chat/completions · Authorization · JSON──▶ provider
your code ◀──200 · JSON · x-ratelimit-remaining ─────────────  provider
```

| Use it when | Don't when | Instead use |
|---|---|---|
| a provider with no SDK, an internal model server, custom transports | an official SDK exists — it handles retries, streaming and error types | the vendor SDK |

**Watch out**
- One `Client` for the whole service. A new client per request opens a new connection each time — 300 ms instead of 50 ms, and eventually no free sockets.
- The default timeout is 5 seconds for everything, which kills long answers. Set all four.
- After `raise_for_status()`, decide per code: 429 → retry with backoff, 401 → stop and alert, 5xx → retry, 400 → fix the request.